# Backtest Example: Simple Market Maker

Runs a volatility-adaptive market making strategy that quotes bid/ask around mid price
with inventory skew.

**Prerequisites:**
- `gnome-backtest` uber JAR built: `cd gnome-backtest && mvn package -DskipTests`
- AWS credentials configured + `aws sso login`

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

import plotly.io as pio
pio.renderers.default = "notebook"

from gnomepy.java import (
    ensure_jvm_started,
    Backtest,
    ExchangeConfig,
    StaticFeeConfig,
    StaticLatencyConfig,
    SchemaType,
    RiskConfig,
    RiskAverseQueueConfig,
    ProbabilisticQueueConfig,
    OptimisticQueueConfig,
)
from gnomepy.strategies import SimpleMarketMaker
from gnomepy.signals import SpreadVolatility
from gnomepy.signals import MicropriceFairValue, WeightedMicropriceFairValue, DampenedFairValue
from gnomepy.reporting import BacktestReport

ensure_jvm_started()

## Configuration

In [ ]:
SECURITY_ID = 2
EXCHANGE_ID = 1
#START = datetime(2026, 1, 20, 12, 0)
#END = datetime(2026, 1, 20, 19, 0)
START = datetime(2026, 1, 23, 10, 0)
END = datetime(2026, 1, 23, 11, 0)

## Strategy Setup

The market maker quotes bid/ask around mid price:
- **Spread** scales with realized volatility (Kalman filter) and adapts to fill rate
- **Skew** shifts quotes away from current inventory to mean-revert position (quadratic)
- **Size taper** reduces quote size on the risky side as inventory grows
- **Active flattening** sends aggressive IOC to reduce excess inventory
- **Fill rate adaptation** widens spread when getting picked off, tightens when quiet

In [ ]:
# Volatility signal: spread-implied vol (adapts to market spread regime)
vol_signal = SpreadVolatility(alpha=0.99, warmup_ticks=50, scale=0.5)

strategy = SimpleMarketMaker(
    security_id=SECURITY_ID,
    exchange_id=EXCHANGE_ID,
    fair_value_model=DampenedFairValue(MicropriceFairValue(), 0.0),
    vol_signal=vol_signal,
    spread_multiplier=2.0,        # spread ≈ scale * market_spread * this
    skew_intensity=0.2,
    order_size=100_000,           # quote size (scaled)
    max_position=500_000,       # position limit (scaled)
    min_spread_bps=0.0,           # floor on half-spread
    flatten_threshold=999.0,        # actively flatten when at 100% of max
    target_fill_rate=10.0,        # desired fills/min for adaptive spread
    fill_rate_window_sec=60.0,    # 1 min lookback
    min_spread_scale=1.0,         # can halve spread when quiet
    max_spread_scale=3.0,         # can triple spread when toxic
)

## Run Backtest

In [ ]:
# Fee model: maker rebate, taker fee
fee_model = StaticFeeConfig(taker_fee=0.00035, maker_fee=-0.0001)

# Latency: 50ms network round-trip, 5ms exchange processing
network_latency = StaticLatencyConfig(latency_nanos=5_000_000)        # 5ms
order_processing_latency = StaticLatencyConfig(latency_nanos=100_000)

backtest = Backtest(
    strategy=strategy,
    schema_type=SchemaType.MBP_10,
    start_date=START,
    end_date=END,
    exchanges=[
        ExchangeConfig(
            exchange_id=EXCHANGE_ID,
            security_id=SECURITY_ID,
            fee_model=fee_model,
            network_latency=network_latency,
            order_processing_latency=order_processing_latency,
            queue_model=RiskAverseQueueConfig(),
        ),
    ],
    risk_config=RiskConfig(
        default_tick_size=10_000_000,
        tick_sizes={1: 100_000_000, 2: 10_000_000}
    ),
)

results = backtest.run()
report = BacktestReport(results)

print(f"Market records: {results.market_record_count}")
print(f"Execution records: {results.execution_record_count}")
print(f"Intent records: {results.intent_record_count}")

## Summary

In [ ]:
summary = report.summary()
for k, v in summary.items():
    print(f"{k}: {v}")

## Price & Fills

In [ ]:
report.plot_price()

## Position

In [ ]:
report.plot_position()

## PnL

In [ ]:
report.plot_pnl()

## PnL Decomposition

In [ ]:
report.plot_pnl_decomposition()

## Drawdown

In [ ]:
report.plot_drawdown()

## Spread & Fees

In [ ]:
report.plot_spread()

In [ ]:
report.plot_fees()

## Fill Scatter

In [ ]:
report.plot_fills()

## Export HTML Report

In [ ]:
report.generate_html("/tmp/backtest_mm_report.html")
print("Report saved to /tmp/backtest_mm_report.html")

## Raw Data Access

In [ ]:
# Execution records
exec_df = results.execution_records_df()
print(f"Executions: {len(exec_df)}")
exec_df.head(10)

In [ ]:
# Intent records — what the strategy wanted to quote each tick
intent_df = results.intent_records_df()
print(f"Intents: {len(intent_df)}")
print(f"Intents with takes: {(intent_df['take_size'] > 0).sum()}")
intent_df.tail(10)

In [ ]:
report.quote_quality()

In [ ]:
report.plot_quote_vs_spread()

In [ ]:
fq = report.fill_quality()
fq.head(10)
import pandas as pd
# Spread capture by distance bucket
fq["distance_bucket"] = pd.cut(fq["distance"], bins=[-100, -1, 0, 1, 5, 10, 50, 1000])
print(fq.groupby("distance_bucket").agg(
  count=("spread_capture", "count"),
  total_sc=("spread_capture", "sum"),
  avg_sc=("spread_capture", "mean"),
  avg_distance=("distance", "mean"),
).to_string())